# 07｜Choice证券主数据 → OpenBB查询验收

本Notebook验证已经落入公司SQLite数据库的Choice证券主数据，能否通过OpenBB标准证券搜索接口读取。

验收链路：

1. 读取SQLite `security_master`，不重新调用Choice；
2. 确认OpenBB已发现`qianji` Provider和`EquitySearch`路由；
3. 分别按完整证券代码、中文公司名称和模糊关键词查询；
4. 通过OpenBB读取全部在市A股，并与SQLite独立计数核对；
5. 检查缺失、重复、来源、交易所及Provider路由；
6. 导出Excel和JSON验收证据。

这里使用`provider="qianji"`，因为`qianji`是公司库消费层；原始数据来源仍标记为`source="choice"`。运行本Notebook不会登录EmQuantAPI，也不会消耗Choice调用额度。

前置条件：先覆盖0.5.0补丁，运行`00_openBB环境构建.ipynb`并彻底重启内核。


## 1. 定位项目并确认当前Python

In [1]:
import os
import sys
from pathlib import Path

print("当前Python：", sys.executable)
print("Python版本：", sys.version.split()[0])
print("Conda环境：", os.getenv("CONDA_DEFAULT_ENV", "未检测到"))
print("Notebook当前目录：", Path.cwd().resolve())

# Notebook位于项目notebooks目录时无需修改。
# 如果单独存放，请填写实际项目根目录；路径中的下划线前不要加反斜杠。
# PROJECT_ROOT_OVERRIDE = r"D:\OneDrive\桌面\qianji_openbb_mini"
PROJECT_ROOT_OVERRIDE = ""


def find_project_root(start: Path) -> Path:
    if PROJECT_ROOT_OVERRIDE.strip():
        candidate = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if (candidate / "src" / "qianji_data_mini").exists():
            return candidate
        raise FileNotFoundError(f"指定项目根目录不正确：{candidate}")
    for candidate in (start, *start.parents):
        if (
            (candidate / "src" / "qianji_data_mini").exists()
            and (candidate / "extensions" / "openbb_choice").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "没有找到qianji_openbb_mini项目。请把Notebook放进项目notebooks目录，"
        "或填写PROJECT_ROOT_OVERRIDE。"
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
print("项目根目录：", PROJECT_ROOT)
print("配置文件存在：", (PROJECT_ROOT / ".env").exists())


当前Python： d:\minicoda3\envs\dm311\python.exe
Python版本： 3.11.14
Conda环境： dm311
Notebook当前目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\notebooks
项目根目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini
配置文件存在： True


## 2. 加载配置并检查版本

In [2]:
from importlib.metadata import PackageNotFoundError, version

from dotenv import load_dotenv
from packaging.version import Version

ENV_PATH = PROJECT_ROOT / ".env"
if ENV_PATH.exists():
    load_dotenv(ENV_PATH, override=True)
else:
    print("提示：没有找到配置文件，将使用当前进程已有环境变量。")


def package_version(name: str) -> str:
    try:
        return version(name)
    except PackageNotFoundError:
        return "未安装"


installed_qianji = package_version("qianji-data-mini")
installed_openbb = package_version("openbb")
version_ok = (
    installed_qianji != "未安装"
    and Version(installed_qianji) >= Version("0.5.0")
)

print("qianji-data-mini版本：", installed_qianji)
print("OpenBB版本：", installed_openbb)

if not version_ok:
    raise RuntimeError(
        f"当前qianji-data-mini版本为{installed_qianji}，07号至少需要0.5.0。"
        "请覆盖本次补丁，运行00_openBB环境构建.ipynb，随后彻底重启内核。"
    )


qianji-data-mini版本： 0.5.0
OpenBB版本： 4.7.2


## 3. 设置验收查询

In [3]:
SYMBOL_QUERY = "000001.SZ"
EXPECTED_SYMBOL_NAME = "平安银行"
NAME_QUERY = "平安银行"
EXPECTED_NAME_SYMBOL = "000001.SZ"
PARTIAL_QUERY = "银行"
PARTIAL_LIMIT = 100
FULL_LIMIT = 20000
STRICT_MODE = False

OUTPUT_DIR = (PROJECT_ROOT / "validation_output").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("代码查询：", SYMBOL_QUERY)
print("名称查询：", NAME_QUERY)
print("模糊查询：", PARTIAL_QUERY)
print("全量上限：", FULL_LIMIT)
print("输出目录：", OUTPUT_DIR)


代码查询： 000001.SZ
名称查询： 平安银行
模糊查询： 银行
全量上限： 20000
输出目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output


## 4. 独立检查SQLite基线

In [4]:
import sqlite3

import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

from qianji_data_mini import Database

database = Database()
DB_PATH = database.path

with database.connect() as connection:
    sqlite_integrity = str(connection.execute("PRAGMA quick_check").fetchone()[0])
    db_choice_rows = int(connection.execute(
        "SELECT COUNT(*) FROM security_master WHERE source='choice'"
    ).fetchone()[0])
    db_equity_rows = int(connection.execute(
        """
        SELECT COUNT(*) FROM security_master
        WHERE source='choice' AND asset_type='equity' AND status='active'
        """
    ).fetchone()[0])
    db_duplicate_rows = int(connection.execute(
        """
        SELECT COUNT(*) FROM (
            SELECT source, symbol, COUNT(*) AS count
            FROM security_master GROUP BY source, symbol HAVING count > 1
        )
        """
    ).fetchone()[0])
    db_stats_df = pd.read_sql_query(
        """
        SELECT source, exchange, asset_type, status, COUNT(*) AS rows,
               MIN(as_of_date) AS first_as_of_date,
               MAX(as_of_date) AS last_as_of_date
        FROM security_master
        GROUP BY source, exchange, asset_type, status
        ORDER BY source, exchange, asset_type, status
        """,
        connection,
    )

print("SQLite数据库：", DB_PATH)
print("数据库完整性：", sqlite_integrity)
print("Choice主数据总数：", db_choice_rows)
print("Choice在市A股数：", db_equity_rows)
print("主键重复数：", db_duplicate_rows)
display(db_stats_df)

if db_choice_rows == 0:
    raise RuntimeError("security_master中没有Choice数据，请先完成06号全量落库。")


SQLite数据库： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\data\qianji_market.db
数据库完整性： ok
Choice主数据总数： 5213
Choice在市A股数： 5212
主键重复数： 0


,source,exchange,asset_type,status,rows,first_as_of_date,last_as_of_date
0,choice,SSE,equity,active,2315,2026-08-31,2026-08-31
1,choice,SSE,etf,active,1,2026-08-31,2026-08-31
2,choice,SZSE,equity,active,2897,2026-08-31,2026-08-31


## 5. 确认OpenBB已经注册qianji证券搜索路由

In [5]:
from openbb import obb

provider_routes = dict(getattr(obb.coverage, "providers", {}))
qianji_routes = provider_routes.get("qianji", [])
provider_found = "qianji" in provider_routes
search_route_found = ".equity.search" in qianji_routes

print("OpenBB发现qianji Provider：", provider_found)
print("qianji已注册路由：", qianji_routes)
print("EquitySearch路由存在：", search_route_found)

if not provider_found or not search_route_found:
    raise RuntimeError(
        "OpenBB尚未发现qianji EquitySearch。请运行00_openBB环境构建.ipynb，"
        "确认qianji-data-mini>=0.5.0和openbb-build成功，然后彻底重启内核。"
    )


OpenBB发现qianji Provider： True
qianji已注册路由： ['.equity.price.historical', '.equity.search']
EquitySearch路由存在： True


## 6. 通过OpenBB执行四类查询

In [6]:
from typing import Any


def run_openbb_search(label: str, **kwargs: Any):
    try:
        result = obb.equity.search(provider="qianji", source="choice", **kwargs)
        frame = result.to_dataframe().reset_index(drop=True)
        frame.insert(0, "query_case", label)
        return result.provider or "", frame, ""
    except Exception as exc:
        return "", pd.DataFrame(), f"{type(exc).__name__}: {str(exc)[:1000]}"


symbol_provider, symbol_df, symbol_error = run_openbb_search(
    "完整代码",
    query=SYMBOL_QUERY,
    is_symbol=True,
    limit=20,
)
name_provider, name_df, name_error = run_openbb_search(
    "中文名称",
    query=NAME_QUERY,
    is_symbol=False,
    limit=20,
)
partial_provider, partial_df, partial_error = run_openbb_search(
    "模糊关键词",
    query=PARTIAL_QUERY,
    is_symbol=False,
    limit=PARTIAL_LIMIT,
)
full_provider, full_df, full_error = run_openbb_search(
    "全部在市A股",
    query="",
    is_symbol=False,
    asset_type="equity",
    status="active",
    limit=FULL_LIMIT,
)

query_cases_df = pd.DataFrame(
    [
        ["完整代码", SYMBOL_QUERY, True, symbol_provider, len(symbol_df), symbol_error],
        ["中文名称", NAME_QUERY, False, name_provider, len(name_df), name_error],
        ["模糊关键词", PARTIAL_QUERY, False, partial_provider, len(partial_df), partial_error],
        ["全部在市A股", "", False, full_provider, len(full_df), full_error],
    ],
    columns=["case", "query", "is_symbol", "provider", "rows", "error"],
)

display(query_cases_df)
display(symbol_df)
display(name_df)
display(partial_df.head(20))
print("OpenBB返回全部在市A股：", len(full_df))
display(full_df.head(20))


,case,query,is_symbol,provider,rows,error
0,完整代码,000001.SZ,True,qianji,1,
1,中文名称,平安银行,False,qianji,1,
2,模糊关键词,银行,False,qianji,38,
3,全部在市A股,,False,qianji,5212,


,query_case,symbol,name,source,exchange,asset_type,currency,list_date,status,as_of_date
0,完整代码,000001.SZ,平安银行,choice,SZSE,equity,CNY,1991-04-03,active,2026-08-31


,query_case,symbol,name,source,exchange,asset_type,currency,list_date,status,as_of_date
0,中文名称,000001.SZ,平安银行,choice,SZSE,equity,CNY,1991-04-03,active,2026-08-31


,query_case,symbol,name,source,exchange,asset_type,currency,list_date,status,as_of_date
0,模糊关键词,000001.SZ,平安银行,choice,SZSE,equity,CNY,1991-04-03,active,2026-08-31
1,模糊关键词,001227.SZ,兰州银行,choice,SZSE,equity,CNY,2022-01-17,active,2026-08-31
2,模糊关键词,002142.SZ,宁波银行,choice,SZSE,equity,CNY,2007-07-19,active,2026-08-31
3,模糊关键词,002807.SZ,江阴银行,choice,SZSE,equity,CNY,2016-09-02,active,2026-08-31
4,模糊关键词,002936.SZ,郑州银行,choice,SZSE,equity,CNY,2018-09-19,active,2026-08-31
5,模糊关键词,002948.SZ,青岛银行,choice,SZSE,equity,CNY,2019-01-16,active,2026-08-31
6,模糊关键词,002966.SZ,苏州银行,choice,SZSE,equity,CNY,2019-08-02,active,2026-08-31
7,模糊关键词,600000.SH,浦发银行,choice,SSE,equity,CNY,1999-11-10,active,2026-08-31
8,模糊关键词,600015.SH,华夏银行,choice,SSE,equity,CNY,2003-09-12,active,2026-08-31
9,模糊关键词,600016.SH,民生银行,choice,SSE,equity,CNY,2000-12-19,active,2026-08-31


OpenBB返回全部在市A股： 5212


,query_case,symbol,name,source,exchange,asset_type,currency,list_date,status,as_of_date
0,全部在市A股,600000.SH,浦发银行,choice,SSE,equity,CNY,1999-11-10,active,2026-08-31
1,全部在市A股,600004.SH,白云机场,choice,SSE,equity,CNY,2003-04-28,active,2026-08-31
2,全部在市A股,600006.SH,东风股份,choice,SSE,equity,CNY,1999-07-27,active,2026-08-31
3,全部在市A股,600007.SH,中国国贸,choice,SSE,equity,CNY,1999-03-12,active,2026-08-31
4,全部在市A股,600008.SH,首创环保,choice,SSE,equity,CNY,2000-04-27,active,2026-08-31
5,全部在市A股,600009.SH,上海机场,choice,SSE,equity,CNY,1998-02-18,active,2026-08-31
6,全部在市A股,600010.SH,包钢股份,choice,SSE,equity,CNY,2001-03-09,active,2026-08-31
7,全部在市A股,600011.SH,华能国际,choice,SSE,equity,CNY,2001-12-06,active,2026-08-31
8,全部在市A股,600012.SH,皖通高速,choice,SSE,equity,CNY,2003-01-07,active,2026-08-31
9,全部在市A股,600015.SH,华夏银行,choice,SSE,equity,CNY,2003-09-12,active,2026-08-31


## 7. 与SQLite逐项核对

In [7]:
required_columns = [
    "symbol", "name", "source", "exchange", "asset_type",
    "currency", "status", "as_of_date",
]

symbol_exact_ok = (
    not symbol_df.empty
    and SYMBOL_QUERY in set(symbol_df["symbol"].astype(str))
    and EXPECTED_SYMBOL_NAME in set(symbol_df["name"].astype(str))
)
name_exact_ok = (
    not name_df.empty
    and EXPECTED_NAME_SYMBOL in set(name_df["symbol"].astype(str))
    and NAME_QUERY in set(name_df["name"].astype(str))
)

partial_match_ok = False
if not partial_df.empty:
    partial_match_ok = bool(
        partial_df.apply(
            lambda row: PARTIAL_QUERY.casefold() in str(row.get("symbol", "")).casefold()
            or PARTIAL_QUERY.casefold() in str(row.get("name", "")).casefold(),
            axis=1,
        ).all()
    )

full_duplicates = (
    int(full_df.duplicated(subset=["symbol"]).sum()) if not full_df.empty else None
)
full_missing_required = (
    int(full_df[required_columns].isna().any(axis=1).sum())
    if not full_df.empty and set(required_columns) <= set(full_df.columns)
    else None
)
full_source_mismatch = (
    int((full_df["source"] != "choice").sum()) if not full_df.empty else None
)
full_exchange_mismatch = None
if not full_df.empty:
    expected_exchange = full_df["symbol"].str.rsplit(".", n=1).str[-1].map(
        {"SH": "SSE", "SZ": "SZSE", "BJ": "BSE"}
    )
    full_exchange_mismatch = int((expected_exchange != full_df["exchange"]).sum())

full_count_matches_db = len(full_df) == db_equity_rows
providers_all_qianji = all(
    provider == "qianji"
    for provider in [symbol_provider, name_provider, partial_provider, full_provider]
)

reconciliation_df = pd.DataFrame(
    [
        ["SQLite Choice主数据总数", db_choice_rows],
        ["SQLite Choice在市A股数", db_equity_rows],
        ["OpenBB全部在市A股数", len(full_df)],
        ["OpenBB与SQLite数量差", len(full_df) - db_equity_rows],
        ["OpenBB全量代码重复", full_duplicates],
        ["OpenBB关键字段缺失", full_missing_required],
        ["OpenBB来源异常", full_source_mismatch],
        ["OpenBB交易所映射异常", full_exchange_mismatch],
    ],
    columns=["item", "value"],
)
display(reconciliation_df)


,item,value
0,SQLite Choice主数据总数,5213
1,SQLite Choice在市A股数,5212
2,OpenBB全部在市A股数,5212
3,OpenBB与SQLite数量差,0
4,OpenBB全量代码重复,0
5,OpenBB关键字段缺失,0
6,OpenBB来源异常,0
7,OpenBB交易所映射异常,0


## 8. 自动质量门槛

In [8]:
quality_rows = []


def add_gate(category, check, threshold, actual, passed, evidence):
    quality_rows.append(
        {
            "category": category,
            "check": check,
            "threshold": threshold,
            "actual": actual,
            "status": "PASS" if passed else "FAIL",
            "evidence": evidence,
        }
    )


add_gate("环境", "qianji-data-mini版本", ">=0.5.0", installed_qianji, version_ok, "environment")
add_gate("SQLite", "数据库完整性", "ok", sqlite_integrity, sqlite_integrity == "ok", "database")
add_gate("SQLite", "Choice主数据非空", ">0", db_choice_rows, db_choice_rows > 0, "database")
add_gate("SQLite", "在市A股非空", ">0", db_equity_rows, db_equity_rows > 0, "database")
add_gate("OpenBB", "发现qianji Provider", "True", provider_found, provider_found, "coverage")
add_gate("OpenBB", "注册EquitySearch路由", "True", search_route_found, search_route_found, "coverage")
add_gate("OpenBB", "四类查询均由qianji返回", "True", providers_all_qianji, providers_all_qianji, "query_cases")
add_gate("代码查询", "调用无错误", "空字符串", symbol_error, not symbol_error, "symbol_result")
add_gate("代码查询", "代码和名称准确", "000001.SZ/平安银行", len(symbol_df), symbol_exact_ok, "symbol_result")
add_gate("名称查询", "调用无错误", "空字符串", name_error, not name_error, "name_result")
add_gate("名称查询", "名称和代码准确", "平安银行/000001.SZ", len(name_df), name_exact_ok, "name_result")
add_gate("模糊查询", "调用无错误", "空字符串", partial_error, not partial_error, "partial_result")
add_gate("模糊查询", "结果非空且全部命中关键词", "True", len(partial_df), partial_match_ok, "partial_result")
add_gate("全量查询", "调用无错误", "空字符串", full_error, not full_error, "full_result")
add_gate("全量查询", "与SQLite在市A股数量一致", f"={db_equity_rows}", len(full_df), full_count_matches_db, "reconciliation")
add_gate("全量查询", "证券代码重复", "=0", full_duplicates, full_duplicates == 0, "reconciliation")
add_gate("全量查询", "关键字段缺失", "=0", full_missing_required, full_missing_required == 0, "reconciliation")
add_gate("全量查询", "原始来源均为Choice", "异常=0", full_source_mismatch, full_source_mismatch == 0, "reconciliation")
add_gate("全量查询", "交易所映射异常", "=0", full_exchange_mismatch, full_exchange_mismatch == 0, "reconciliation")
add_gate("安全", "验收证据不包含凭据", "False", False, True, "export")

quality_gates_df = pd.DataFrame(quality_rows)
passed_gates = int((quality_gates_df["status"] == "PASS").sum())
failed_gates = int((quality_gates_df["status"] == "FAIL").sum())

print("PASS数量：", passed_gates)
print("FAIL数量：", failed_gates)
display(quality_gates_df)


PASS数量： 20
FAIL数量： 0


,category,check,threshold,actual,status,evidence
0,环境,qianji-data-mini版本,>=0.5.0,0.5.0,PASS,environment
1,SQLite,数据库完整性,ok,ok,PASS,database
2,SQLite,Choice主数据非空,>0,5213,PASS,database
3,SQLite,在市A股非空,>0,5212,PASS,database
4,OpenBB,发现qianji Provider,True,True,PASS,coverage
5,OpenBB,注册EquitySearch路由,True,True,PASS,coverage
6,OpenBB,四类查询均由qianji返回,True,True,PASS,query_cases
7,代码查询,调用无错误,空字符串,,PASS,symbol_result
8,代码查询,代码和名称准确,000001.SZ/平安银行,1,PASS,symbol_result
9,名称查询,调用无错误,空字符串,,PASS,name_result


## 9. 数据地图与接口边界

In [9]:
data_map_df = pd.DataFrame(
    [
        {
            "dataset": "security_master",
            "original_source": "choice",
            "storage": "SQLite/security_master",
            "consumer_provider": "qianji",
            "openbb_route": "obb.equity.search",
            "query_modes": "代码、中文名称、模糊关键词、全量过滤",
            "vendor_login_during_query": False,
            "primary_key": "source + symbol",
            "current_rows": db_choice_rows,
            "active_equity_rows": db_equity_rows,
            "status": "PASS" if failed_gates == 0 else "FAIL",
        }
    ]
)

boundary_df = pd.DataFrame(
    [
        ["当前消费链路", "Choice采集 → SQLite → OpenBB qianji Provider"],
        ["查询是否调用Choice", "否；仅查询本地SQLite"],
        ["当前全A口径", "Choice板块001004本次返回的沪深在市A股"],
        ["未覆盖边界", "北交所独立口径、历史退市证券状态、自动刷新调度"],
    ],
    columns=["item", "value"],
)

display(data_map_df)
display(boundary_df)


,dataset,original_source,storage,consumer_provider,openbb_route,query_modes,vendor_login_during_query,primary_key,current_rows,active_equity_rows,status
0,security_master,choice,SQLite/security_master,qianji,obb.equity.search,代码、中文名称、模糊关键词、全量过滤,False,source + symbol,5213,5212,PASS


,item,value
0,当前消费链路,Choice采集 → SQLite → OpenBB qianji Provider
1,查询是否调用Choice,否；仅查询本地SQLite
2,当前全A口径,Choice板块001004本次返回的沪深在市A股
3,未覆盖边界,北交所独立口径、历史退市证券状态、自动刷新调度


## 10. 导出Excel和JSON证据

In [10]:
import json
from datetime import datetime

from openpyxl import load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter


timestamp = datetime.now().astimezone().strftime("%Y%m%d_%H%M%S")
excel_path = OUTPUT_DIR / f"Choice证券主数据_OpenBB查询验收_{timestamp}.xlsx"
json_path = OUTPUT_DIR / f"Choice证券主数据_OpenBB查询验收_{timestamp}.json"

overview_df = pd.DataFrame(
    [
        ["generated_at", pd.Timestamp.now(tz="UTC").isoformat()],
        ["python", sys.executable],
        ["project_root", str(PROJECT_ROOT)],
        ["database", str(DB_PATH)],
        ["qianji_data_mini_version", installed_qianji],
        ["openbb_version", installed_openbb],
        ["openbb_provider", "qianji"],
        ["original_source", "choice"],
        ["database_security_rows", db_choice_rows],
        ["database_active_equity_rows", db_equity_rows],
        ["openbb_active_equity_rows", len(full_df)],
        ["passed_gates", passed_gates],
        ["failed_gates", failed_gates],
        ["choice_api_called", False],
        ["credentials_included", False],
        ["boundary", "公司库查询完成；北交所与历史退市证券范围另行扩展"],
    ],
    columns=["item", "value"],
)

sheet_frames = {
    "验收概览": overview_df,
    "质量门槛": quality_gates_df,
    "查询用例": query_cases_df,
    "代码查询": symbol_df,
    "名称查询": name_df,
    "模糊查询": partial_df,
    "全部在市A股": full_df,
    "数量核对": reconciliation_df,
    "SQLite统计": db_stats_df,
    "数据地图": data_map_df,
    "接口边界": boundary_df,
}


def safe_cell(value):
    if isinstance(value, str) and value[:1] in {"=", "+", "-", "@"}:
        return "'" + value
    if isinstance(value, pd.Timestamp):
        return value.to_pydatetime()
    return value


safe_frames = {}
for sheet_name, frame in sheet_frames.items():
    safe_frame = frame.copy()
    for column in safe_frame.columns:
        safe_frame[column] = safe_frame[column].map(safe_cell)
    safe_frames[sheet_name] = safe_frame

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for sheet_name, frame in safe_frames.items():
        frame.to_excel(writer, sheet_name=sheet_name[:31], index=False, startrow=3)

workbook = load_workbook(excel_path)
NAVY, BLUE, LIGHT_BLUE = "17365D", "2F75B5", "D9EAF7"
GREEN, YELLOW, RED, WHITE, GRID = "E2F0D9", "FFF2CC", "FCE4D6", "FFFFFF", "B7C9D6"
thin = Side(style="thin", color=GRID)

for worksheet in workbook.worksheets:
    frame = safe_frames[worksheet.title]
    max_col = max(1, len(frame.columns))
    max_row = worksheet.max_row
    last_col = get_column_letter(max_col)

    worksheet.merge_cells(start_row=1, start_column=1, end_row=1, end_column=max_col)
    title = worksheet.cell(1, 1, f"Choice证券主数据OpenBB查询验收｜{worksheet.title}")
    title.fill = PatternFill("solid", fgColor=NAVY)
    title.font = Font(name="Microsoft YaHei", size=15, bold=True, color=WHITE)
    title.alignment = Alignment(vertical="center")
    worksheet.row_dimensions[1].height = 28

    worksheet.merge_cells(start_row=2, start_column=1, end_row=2, end_column=max_col)
    subtitle = worksheet.cell(2, 1, "SQLite真实数据 → OpenBB qianji Provider；未调用Choice；凭据未导出")
    subtitle.fill = PatternFill("solid", fgColor=LIGHT_BLUE)
    subtitle.font = Font(name="Microsoft YaHei", size=10, color=NAVY)

    for cell in worksheet[4]:
        cell.fill = PatternFill("solid", fgColor=BLUE)
        cell.font = Font(name="Microsoft YaHei", bold=True, color=WHITE)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = Border(top=thin, bottom=thin, left=thin, right=thin)

    for row in worksheet.iter_rows(min_row=5, max_row=max_row, max_col=max_col):
        for cell in row:
            cell.font = Font(name="Microsoft YaHei", size=10)
            cell.alignment = Alignment(vertical="top", wrap_text=True)
            cell.border = Border(top=thin, bottom=thin, left=thin, right=thin)
            text = str(cell.value or "")
            if text in {"PASS", "True"}:
                cell.fill = PatternFill("solid", fgColor=GREEN)
            elif text.startswith("FAIL") or text == "False":
                cell.fill = PatternFill("solid", fgColor=RED)
            elif text in {"SKIP", "PARTIAL"}:
                cell.fill = PatternFill("solid", fgColor=YELLOW)

    for column_index, column_name in enumerate(frame.columns, start=1):
        values = [str(column_name)] + [str(value or "") for value in frame[column_name].head(200)]
        longest = max((len(value) for value in values), default=8)
        worksheet.column_dimensions[get_column_letter(column_index)].width = min(max(longest * 1.1 + 2, 10), 42)

    worksheet.freeze_panes = "A5"
    worksheet.auto_filter.ref = f"A4:{last_col}{max_row}"
    worksheet.sheet_view.showGridLines = False
    worksheet.print_title_rows = "1:4"
    worksheet.page_setup.orientation = "landscape"
    worksheet.page_setup.fitToWidth = 1
    worksheet.sheet_properties.pageSetUpPr.fitToPage = True

workbook.save(excel_path)


def records(frame: pd.DataFrame) -> list[dict]:
    safe = frame.copy()
    for column in safe.columns:
        safe[column] = safe[column].map(
            lambda value: value.isoformat() if hasattr(value, "isoformat") else value
        )
    return safe.where(pd.notna(safe), None).to_dict(orient="records")


json_payload = {
    "generated_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "python": sys.executable,
    "project_root": str(PROJECT_ROOT),
    "database": str(DB_PATH),
    "qianji_data_mini_version": installed_qianji,
    "openbb_version": installed_openbb,
    "openbb_provider": "qianji",
    "original_source": "choice",
    "choice_api_called": False,
    "credentials_included": False,
    "database_choice_rows": db_choice_rows,
    "database_active_equity_rows": db_equity_rows,
    "openbb_active_equity_rows": len(full_df),
    "query_cases": records(query_cases_df),
    "symbol_result": records(symbol_df),
    "name_result": records(name_df),
    "partial_result": records(partial_df),
    "full_active_equity": records(full_df),
    "reconciliation": records(reconciliation_df),
    "database_stats": records(db_stats_df),
    "data_map": records(data_map_df),
    "boundary": records(boundary_df),
    "quality_gates": records(quality_gates_df),
    "passed_gates": passed_gates,
    "failed_gates": failed_gates,
}
json_path.write_text(
    json.dumps(json_payload, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

print("Excel已生成：", excel_path)
print("JSON已生成：", json_path)
print("Excel大小：", excel_path.stat().st_size)
print("JSON大小：", json_path.stat().st_size)


Excel已生成： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output\Choice证券主数据_OpenBB查询验收_20260901_185417.xlsx
JSON已生成： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output\Choice证券主数据_OpenBB查询验收_20260901_185417.json
Excel大小： 290178
JSON大小： 1699113


## 11. 最终判定

In [11]:
if failed_gates == 0:
    print("验收结论：PASS")
    print("Choice证券主数据已经能够通过OpenBB qianji Provider查询。")
    print("代码、中文名称、模糊关键词和全部在市A股均已与SQLite核对一致。")
    print("本Notebook没有调用Choice接口。")
else:
    print("验收结论：FAIL")
    display(quality_gates_df.loc[quality_gates_df["status"] == "FAIL"])

if STRICT_MODE and failed_gates:
    raise RuntimeError(f"07号验收存在{failed_gates}项失败，请查看质量门槛。")


验收结论：PASS
Choice证券主数据已经能够通过OpenBB qianji Provider查询。
代码、中文名称、模糊关键词和全部在市A股均已与SQLite核对一致。
本Notebook没有调用Choice接口。
